In [26]:
import json
import pandas as pd
import time
import re
import ast
import requests
import shutil
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support import expected_conditions as EC

In [8]:
# df view settings
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [9]:
# Find the actual binaries
CHROME_BINARY = shutil.which("chromium")
CHROMEDRIVER_PATH = shutil.which("chromedriver")

chrome_options = Options()
chrome_options.binary_location = CHROME_BINARY

chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

service = Service(CHROMEDRIVER_PATH)

driver = webdriver.Chrome(service=service, options=chrome_options)


Chromium binary: /usr/bin/chromium
Chromedriver: /usr/bin/chromedriver


In [10]:
# retrieving all the distinct car brands
u = "https://www.bilbasen.dk/brugt/bil?includeengroscvr=true&includeleasing=false"
data = json.loads(
    BeautifulSoup(requests.get(u, headers={"User-Agent": "Mozilla/5.0"}).text, "html.parser")
    .find("script", id="__NEXT_DATA__").string
)

c = []

def walk(x):
    if isinstance(x, list):
        labels = []
        for v in x:
            if isinstance(v, str):
                labels.append(v.strip())
            elif isinstance(v, dict):
                for k in ("label", "name", "title", "text", "value", "displayName"):
                    s = v.get(k)
                    if isinstance(s, str):
                        labels.append(s.strip())
                        break
        if len(labels) >= 30:
            uniq = sorted(set(labels))
            good = [
                s for s in uniq
                if s and len(s) <= 30 and s[0].isalpha() and s[0].isupper()
                and not any(ch.isdigit() for ch in s)
            ]
            if len(good) / len(uniq) > 0.8:
                c.append(uniq)
        for v in x:
            walk(v)
    elif isinstance(x, dict):
        for v in x.values():
            walk(v)

walk(data)

car_brands = sorted(c, key=len, reverse=True)[1]
#driver.quit()

In [11]:
fuel_options = {
    1: 'Benzin',
    2: 'Diesel',
    3: 'El',
    6: 'Hybrid - Benzin',
    8: 'Hybrid - Diesel',
    11: 'Plug-in Benzin',
    12: 'Plug-in Diesel'
}

In [34]:
selected_fuel_type = 1

In [13]:
def download_brand(brand: str, selected_fuel_type: str):
    page_listings = []
    base_url = (
        f"https://www.bilbasen.dk/brugt/bil/{brand}"
        f"?fuel={selected_fuel_type}&includeengroscvr=true&includeleasing=false"
    )
    driver.get(base_url)
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "span[data-e2e='pagination-total']"))
        )
    except TimeoutException:
        print(f"Timeout waiting for pagination on: {brand}")
        return None

    soup1 = BeautifulSoup(driver.page_source, "html.parser")
    page_tag = soup1.find('span', {'data-e2e': 'pagination-total'})
    if not (page_tag and page_tag.text.isdigit()):
        print(f"→ Skipping {brand}: 0 pages found")
        return None

    max_page = int(page_tag.text)

    for page in range(1, max_page + 1):
        paged_url = f"{base_url}&page={page}"
        print(f"Fetching {brand} page {page}/{max_page}")
        driver.get(paged_url)
        soup = BeautifulSoup(driver.page_source, "html.parser")

        for art in soup.find_all("article"):
            if "".join(art.get("class", [])).startswith("Listing_listing"):
                for a in art.find_all("a", class_=lambda c: c and c.startswith("Listing_link")):
                    page_listings.append(a["href"])

    return page_listings

listings = []

for b in car_brands:
    pl = download_brand(b,str(selected_fuel_type))
    if pl:
        listings.extend(pl)

Fetching AC page 1/1
Fetching Abarth page 1/1
Timeout waiting for pagination on: Aiways
Fetching Alfa Romeo page 1/2
Fetching Alfa Romeo page 2/2
Fetching Alpina page 1/1
Fetching Aston Martin page 1/1
Fetching Auburn page 1/1
Fetching Audi page 1/20
Fetching Audi page 2/20
Fetching Audi page 3/20
Fetching Audi page 4/20
Fetching Audi page 5/20
Fetching Audi page 6/20
Fetching Audi page 7/20
Fetching Audi page 8/20
Fetching Audi page 9/20
Fetching Audi page 10/20
Fetching Audi page 11/20
Fetching Audi page 12/20
Fetching Audi page 13/20
Fetching Audi page 14/20
Fetching Audi page 15/20
Fetching Audi page 16/20
Fetching Audi page 17/20
Fetching Audi page 18/20
Fetching Audi page 19/20
Fetching Audi page 20/20
Fetching Austin page 1/1
Fetching Austin Healey page 1/1
Fetching BMW page 1/14
Fetching BMW page 2/14
Fetching BMW page 3/14
Fetching BMW page 4/14
Fetching BMW page 5/14
Fetching BMW page 6/14
Fetching BMW page 7/14
Fetching BMW page 8/14
Fetching BMW page 9/14
Fetching BMW page 

Fetching Nissan page 7/15
Fetching Nissan page 8/15
Fetching Nissan page 9/15
Fetching Nissan page 10/15
Fetching Nissan page 11/15
Fetching Nissan page 12/15
Fetching Nissan page 13/15
Fetching Nissan page 14/15
Fetching Nissan page 15/15
Fetching OScar page 1/1
Fetching Oldsmobile page 1/1
Timeout waiting for pagination on: Omoda
Fetching Opel page 1/26
Fetching Opel page 2/26
Fetching Opel page 3/26
Fetching Opel page 4/26
Fetching Opel page 5/26
Fetching Opel page 6/26
Fetching Opel page 7/26
Fetching Opel page 8/26
Fetching Opel page 9/26
Fetching Opel page 10/26
Fetching Opel page 11/26
Fetching Opel page 12/26
Fetching Opel page 13/26
Fetching Opel page 14/26
Fetching Opel page 15/26
Fetching Opel page 16/26
Fetching Opel page 17/26
Fetching Opel page 18/26
Fetching Opel page 19/26
Fetching Opel page 20/26
Fetching Opel page 21/26
Fetching Opel page 22/26
Fetching Opel page 23/26
Fetching Opel page 24/26
Fetching Opel page 25/26
Fetching Opel page 26/26
Fetching Overland page 1/

Fetching Volvo page 4/7
Fetching Volvo page 5/7
Fetching Volvo page 6/7
Fetching Volvo page 7/7
Timeout waiting for pagination on: Voyah
Fetching Willys page 1/1
Timeout waiting for pagination on: Xpeng
Fetching Yugo page 1/1
Timeout waiting for pagination on: Zeekr
Timeout waiting for pagination on: firefly


In [14]:
print(len(listings), len(set(listings)))

16437 16436


In [15]:
# Getting JSON data from each listing page (avoid navigating tag hierarchies). ~ 1 minute per 100 cars
all_parsed_data = []
total = len(set(listings))
last_report = time.time()

def func_wrapper_for_loop(i, link):
    global last_report

    # Progress monitoring after each 100 pages
    if i % 100 == 0 or i == total:
        now = time.time()
        elapsed = now - last_report
        mins, secs = divmod(int(elapsed), 60)
        print(
            f"{i}/{total} listings done "
            f"({i/total:.1%}) — last batch took {mins}m {secs}s",
            flush=True
        )
        last_report = now

    driver.get(link)
    car_soup = BeautifulSoup(driver.page_source, "html.parser")

    json_text = None
    for s in car_soup.find_all("script"):
        txt = (s.get_text() or "").lstrip()
        if txt.startswith("var _props"):
            m = re.search(r"var\s*_props\s*=\s*({.*?})\s*;", txt, flags=re.DOTALL)
            if m:
                json_text = m.group(1)
                break

    if not json_text:
        print("No _props JSON found on this page " + link)
        return

    try:
        parsed_data = json.loads(json_text)
        all_parsed_data.append(parsed_data)
    except Exception as e:
        print("Error parsing JSON:", e)

for i, link in enumerate(set(listings), start=1):
    func_wrapper_for_loop(i, link)


100/16436 listings done (0.6%) — last batch took 2m 3s
200/16436 listings done (1.2%) — last batch took 2m 10s
300/16436 listings done (1.8%) — last batch took 2m 10s
400/16436 listings done (2.4%) — last batch took 2m 43s
500/16436 listings done (3.0%) — last batch took 2m 18s
600/16436 listings done (3.7%) — last batch took 2m 26s
700/16436 listings done (4.3%) — last batch took 2m 37s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/opel/karl/10-cosmo-5d/6732809
800/16436 listings done (4.9%) — last batch took 2m 50s
900/16436 listings done (5.5%) — last batch took 2m 39s
1000/16436 listings done (6.1%) — last batch took 2m 48s
1100/16436 listings done (6.7%) — last batch took 2m 54s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/peugeot/2008/12-e-thp-110-active-eat6-5d/6492983
1200/16436 listings done (7.3%) — last batch took 2m 58s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/fiat/500c/10-hybrid-dolcevita-2d/6761235
13

8500/16436 listings done (51.7%) — last batch took 2m 37s
8600/16436 listings done (52.3%) — last batch took 2m 40s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/citron/c3/12-puretech-110-shine-5d/6754204
8700/16436 listings done (52.9%) — last batch took 2m 50s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/citron/c3/14-furio-clim-5d/6732243
8800/16436 listings done (53.5%) — last batch took 2m 49s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/vw/up/10-mpi-60-move-up-bmt-5d/6721270
8900/16436 listings done (54.1%) — last batch took 2m 49s
9000/16436 listings done (54.8%) — last batch took 2m 54s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/vw/passat/14-tsi-150-comfortline-premium-variant-dsg-5d/6755293
9100/16436 listings done (55.4%) — last batch took 3m 3s
9200/16436 listings done (56.0%) — last batch took 3m 24s
9300/16436 listings done (56.6%) — last batch took 3m 26s
No _props JSON found on th

No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/ford/c-max/16-titanium-5d/6731416
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/vw/touran/12-tsi-105-comfortline-bmt-7prs-5d/6757060
14500/16436 listings done (88.2%) — last batch took 2m 18s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/hyundai/i10/10-premium-5d/6751891
14600/16436 listings done (88.8%) — last batch took 2m 24s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/skoda/octavia/12-tsi-110-style-combi-5d/6541882
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/hyundai/i30/10-t-gdi-life-komfort-5d/6757712
14700/16436 listings done (89.4%) — last batch took 2m 32s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/ford/b-max/10-scti-100-titanium-5d/6751274
14800/16436 listings done (90.0%) — last batch took 2m 30s
No _props JSON found on this page https://www.bilbasen.dk/brugt/bil/opel/astra/18-16v-140-enjoy-aut

In [16]:
# digest messy JSON data into a flat table of readable data
def extract_name_value(row):
    output = {}
    # Iterate over each cell in the row with its column label.
    for col, cell in row.items():
        # If the cell is a dictionary with the desired keys, transform it.
        if isinstance(cell, dict) and 'name' in cell and 'displayValue' in cell:
            output[cell['name']] = cell['displayValue']
        # If the cell is a string, try to parse it.
        elif isinstance(cell, str):
            try:
                d = ast.literal_eval(cell)
                if isinstance(d, dict) and 'name' in d and 'displayValue' in d:
                    output[d['name']] = d['displayValue']
                else:
                    # Not the desired structure, so keep the original cell under its column name.
                    output[col] = cell
            except Exception:
                # Parsing failed; keep the original cell.
                output[col] = cell
        else:
            # For any other type, simply keep the original cell.
            output[col] = cell
    return pd.Series(output)

In [17]:
all_listings = []

for entry in all_parsed_data:
    # try old key
    listing_data = entry.get("listing")

    # fall back to new path
    if listing_data is None:
        listing_data = []
        for q in (
            entry.get("props", {})
                 .get("pageProps", {})
                 .get("dehydratedState", {})
                 .get("queries", [])
        ):
            listing_data.extend(q.get("state", {}).get("data", {}).get("listings", []))

    if listing_data:
        # keep one level of nesting: 'vehicle.modelInformation' stays a dict
        flat = pd.json_normalize(listing_data, sep=".", max_level=1)
        all_listings.append(flat)

all_listings = pd.concat(all_listings, ignore_index=True)

In [18]:
all_listings = []

for entry in all_parsed_data:
    listing_data = entry.get('listing', {}) # access key values
    if listing_data:  # skip empty ones
        flattened = pd.json_normalize(listing_data) # flatten JSON data into flat table
        all_listings.append(flattened)

# Combine all the flattened listings into one DataFrame
all_listings = pd.concat(all_listings, ignore_index=True)

In [19]:
# Unpacking nested dictionaries into separate columns
df_model_info = all_listings['vehicle.modelInformation'].apply(pd.Series)
df_vehicle_details = all_listings['vehicle.details'].apply(pd.Series)
df_ratings = all_listings['vehicle.ratings.subRatings'].apply(pd.Series)
df_base = all_listings.drop(['vehicle.modelInformation', 'vehicle.details', 'vehicle.ratings.subRatings'], axis=1)
df_expanded = pd.concat([df_base, df_model_info, df_vehicle_details], axis=1)

In [20]:
rows = [extract_name_value(row) for _, row in df_expanded.iterrows()]
df_result = pd.DataFrame(rows)

<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal


In [41]:
benzin_cols = [
        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering', 
        'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Geartype', 'Antal gear', 'Trækvægt', 'Farve',
        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',
        'Drivmiddel', 'Brændstofforbrug','Cylindre', 'Airbags', 'Tankkapacitet','ABS-bremser', 'ESP', 'Periodisk afgift','CO2 udledning', 'Euronorm', 
        'price.description', 'seller.name', 'seller.address.zipCode', 'seller.address.city', 'seller.sellerOtherItems.numberOfListings',
        'vehicle.ratings.average', 'vehicle.ratings.numberOfReviews', 'canonicalUrl','externalId', 'description'
]

el_cols = [
        'scrape_timestamp','price.displayValue', 'Nypris', 'vehicle.make', 'vehicle.model', 'vehicle.variant', 'vehicle.modelYear', '1. registrering', 
        'Kilometertal', 'Ydelse', 'Acceleration', 'Tophastighed', 'Trækvægt', 'Farve',
        'Kategori', 'Type', 'Bagagerumsstørrelse', 'Vægt', 'Bredde', 'Længde', 'Højde', 'Lasteevne', 'Max. trækvægt m/bremse', 'Trækhjul',
        'Drivmiddel', 'Energiforbrug', 'Batterikapacitet', 'Rækkevidde', 'Hjemmeopladning AC', 'Hurtig opladning DC', 'Opladningstid DC 10-80%',
        'Airbags', 'ABS-bremser', 'ESP', 'Døre', 'Periodisk afgift', 
        'price.description', 'seller.name', 'seller.address.zipCode', 'seller.address.city', 'seller.sellerOtherItems.numberOfListings',
        'vehicle.ratings.average', 'vehicle.ratings.numberOfReviews', 'canonicalUrl','externalId', 'description'
    ]

In [42]:
columns = {
    'Benzin': benzin_cols,
    'Diesel': benzin_cols,
    'El':     el_cols,

}

In [23]:
today = pd.Timestamp.now().replace(microsecond=0)
yesterday = today - pd.Timedelta(days=1)
print(today, yesterday)
df_result.insert(0, 'scrape_timestamp', today)

2025-12-09 22:50:15 2025-12-08 22:50:15


In [43]:
df = df_result[columns[fuel_options[selected_fuel_type]]].copy()

In [44]:
today_str = today.strftime("%Y-%m-%d")

df.to_parquet(
    f"/home/pi-vault/projects/bilbasen_webscraping/data/{fuel_options[selected_fuel_type]}/"
    f"{fuel_options[selected_fuel_type]}_listings_{today_str}.parquet",
    index=True,
    engine="fastparquet",
)


In [46]:
df = pd.read_parquet(f"/home/pi-vault/projects/bilbasen_webscraping/data/Benzin/Benzin_listings_2025-12-09.parquet")

In [45]:
print(f"/home/pi-vault/projects/bilbasen_webscraping/data/{fuel_options[selected_fuel_type]}/"
    f"{fuel_options[selected_fuel_type]}_listings_{today_str}.parquet")

/home/pi-vault/projects/bilbasen_webscraping/data/Benzin/Benzin_listings_2025-12-09.parquet


In [47]:
df

,scrape_timestamp,price.displayValue,Nypris,vehicle.make,vehicle.model,vehicle.variant,vehicle.modelYear,1. registrering,Kilometertal,Ydelse,Acceleration,Tophastighed,Geartype,Antal gear,Trækvægt,Farve,Kategori,Type,Bagagerumsstørrelse,Vægt,Bredde,Længde,Højde,Lasteevne,Max. trækvægt m/bremse,Trækhjul,Drivmiddel,Brændstofforbrug,Cylindre,Airbags,Tankkapacitet,ABS-bremser,ESP,Periodisk afgift,CO2 udledning,Euronorm,price.description,seller.name,seller.address.zipCode,seller.address.city,seller.sellerOtherItems.numberOfListings,vehicle.ratings.average,vehicle.ratings.numberOfReviews,canonicalUrl,externalId,description
index,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,2025-12-09 22:50:15,134.900 kr.,-,VW,Golf VII,"1,0 TSi 110 Comfortline Variant 5d",2018,3/2018,109.000 km,110 hk/200 nm,"10,4 sek.",197 km/t,Manuel,6,1.300 kg,Hvid,Personbil,St.car,605 liter,1.195 kg,179 cm,457 cm,148 cm,655 kg,1.300 kg,Forhjulstræk,Benzin,"(NEDC) 20,4 km/l",3,-,50 l,Ja,Ja,1.460 kr. / år,136 g/km,-,None,Carplace A/S,3400,Hillerød,133.0,4.584000,1883.0,https://www.bilbasen.dk/brugt/bil/vw/golf-vii/10-tsi-110-comfortline-variant-5d/6745488,6745488,"Flot og veludstyret Golf Variant med, Svingbart anhængertræk, Full LED forlygter og meget meget mere.\n\n-16"" alufælge\n-Svingbart anhængertræk\n-LED forlygter\n-Bakkamera\n-Adaptiv fartpilot\n-Tonede ruder i bag\n-Parkeringssensor for og bag\n-App connect (Apple CarPlay)\n\nKørecomputer, multifunktionsrat, læderrat, stofindtræk, højdejust. forsæder, splitbagsæde, 16"" alufælge, el-sidespejle m/varme, led kørelys, fuld led forlygter, mørktonede ruder i bag, træk, svingbart træk (manuel), 2 zone klima, fjernb. centrallås, sædevarme, automatisk start/stop, musikstreaming via bluetooth, navigation, android auto, håndfrit til mobil, apple carplay, bakkamera, parkeringssensor (for), parkeringssensor (bag), esp, automatisk nødbremsesystem, adaptiv fartpilot.\n\n🏦 Vi tilbyder finansiering med og uden udbetaling.\n\nKontakt os, så laver vi hurtigt en beregning. 🖥 carplace.dk - 📞 69 155 888\n\nDen annonceret pris er inkl. leveringsomkostninger, ekskl. nummerpladegebyr kr. 1680,-\n\n"
1,2025-12-09 22:50:15,49.900 kr.,249.883 kr.,Citroën,C4 Picasso,"1,2 PureTech 130 Seduction 5d",2018,11/2017,175.000 km,130 hk/230 nm,"11,2 sek.",201 km/t,Manuel,6,1.470 kg,Koksmetal,Personbil,MPV,537 liter,1.505 kg,183 cm,444 cm,161 cm,400 kg,1.470 kg,Forhjulstræk,Benzin,"(NEDC) 20,0 km/l",3,7,57 l,Ja,Ja,1.460 kr. / år,115 g/km,-,None,MKA Biler ApS,8752,Østbirk,5.0,4.034718,160.0,https://www.bilbasen.dk/brugt/bil/citron/c4-picasso/12-puretech-130-seduction-5d/6733654,6733654,"nysynet, læderrat, multifunktionsrat, el-klapbare sidespejle m/varme, mørktonede ruder i bag, led baglygter, led kørelys, isofix, kurvelys, 2 zone klima, køl i handskerum, sædevarme, fartpilot, automatisk start/stop, musikstreaming via bluetooth, bakkamera, parkeringssensor (for), regnsensor, esp, kørecomputer, stofindtræk, træk, kører rigtig rigtig godt, pæn og velholdt."
2,2025-12-09 22:50:15,149.900 kr.,-,Peugeot,208,"1,2 PureTech 100 Like 5d",2023,6/2023,19.000 km,100 hk/205 nm,"9,9 sek.",188 km/t,Manuel,6,1.200 kg,Gul,Personbil,Halvkombi,309 liter,1.181 kg,177 cm,406 cm,144 cm,414 kg,1.200 kg,Forhjulstræk,Benzin,"(WLTP) 19,6 km/l",3,6,44 l,Ja,Ja,1.260 kr. / år,114 g/km,-,None,"P. Christensen A/S, Stellantis Kolding",6000,Kolding,67.0,4.239939,1274.0,https://www.bilbasen.dk/brugt/bil/peugeot/208/12-puretech-100-like-5d/6630794,6630794,"Leder du efter en stilfuld, økonomisk og sprælsk hatchback, der både ser godt ud og leverer på vejen? Så har vi din næste bil klar!\n\n🔴Udstyrshighlights 🔴\n✅ Bakkamera\n✅ Apple Carplay/ Android Auto\n✅ Multifunktionsrat\n✅ Sædevarme\n\nDerudover kan der også nævnes:\nKørecomputer, læderrat, stofindtræk, dellæderindtræk, 16"" alufælge, el-sidespejle, el-sidespejle m/varme, led kørelys, fuldaut. klima, aircondition, nøglefri adgang, nøglefri tænding, fartpilot, sædevarme, håndfrit til mobil, apple carplay, android auto, usb-